In [2]:
import re
paragraph = "这 是 一段\n被 空 白 拆 开 的 文 本"
cleaned = re.sub(r"\s+", "", paragraph)

print(cleaned)


这是一段被空白拆开的文本


In [8]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time
import os


# ============================================================
# 0. 文件路径
# ============================================================

IMG_PATH = "input.jpg"
ASSET_DIR = "assets"

GLASSES_PATH = os.path.join(ASSET_DIR, "glasses.png")
EARS_PATH = os.path.join(ASSET_DIR, "ears.png")
MASK_PATH = os.path.join(ASSET_DIR, "mask.png")
SCAR_PATH = os.path.join(ASSET_DIR, "scar.png")
STICKER_PATH = os.path.join(ASSET_DIR, "sticker.png")


# ============================================================
# 1. 贴图微调参数
#
# 后面如果只是觉得位置差一点，不要改算法，
# 直接改这一块数字即可。
# ============================================================

# 墨镜：大小由“两眼距离”决定
GLASSES_WIDTH_SCALE = 2.55
GLASSES_Y_OFFSET = 0.04

# 猫耳：大小由“脸宽”决定
EAR_WIDTH_SCALE = 0.50

# 两只耳朵相对于额头中心左右分开的距离
EAR_SIDE_OFFSET = 0.38

# 猫耳整体向头顶方向移动的距离
EAR_UP_OFFSET = 0.06

# 猫耳底部压进头发的程度
# 数值越大，耳朵看起来越像“长在头上”
EAR_BASE_DEPTH = 0.14

# 面罩
MASK_WIDTH_SCALE = 1.10
MASK_Y_OFFSET = 0.02

# 战损贴图
SCAR_WIDTH_SCALE = 0.34

# 自由贴纸默认高度约占图片高度的22%
STICKER_BASE_H_RATIO = 0.22


# ============================================================
# 2. 图片读取与预处理
# ============================================================

def load_rgba(path):
    """
    读取透明PNG。

    普通 cv2.imread() 默认只读取 BGR 三个通道，
    这里使用 IMREAD_UNCHANGED 才能保留第四个 alpha 透明通道。
    """
    picture = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if picture is None:
        raise FileNotFoundError(f"找不到文件：{path}")

    if len(picture.shape) != 3 or picture.shape[2] != 4:
        raise ValueError(f"贴图不是4通道透明PNG：{path}")

    return picture


def crop_transparent(rgba):
    """
    去掉PNG四周完全透明的空白。

    如果不裁透明区域，
    图片中心可能和真正图案中心不一致，
    自动定位时就容易偏。
    """
    alpha = rgba[:, :, 3]
    points = cv2.findNonZero(alpha)

    if points is None:
        return rgba

    x, y, w, h = cv2.boundingRect(points)

    # 稍微留2像素边缘，避免霓虹光晕被裁掉
    x1 = max(0, x - 2)
    y1 = max(0, y - 2)
    x2 = min(rgba.shape[1], x + w + 2)
    y2 = min(rgba.shape[0], y + h + 2)

    return rgba[y1:y2, x1:x2]


def split_ears(rgba):
    """
    ears.png 中左右两个耳朵在同一张图。

    这里从中间切成：
    左耳图片 + 右耳图片

    后面分别计算位置，比把两只耳朵作为一个整体定位准确很多。
    """
    h, w = rgba.shape[:2]

    left = rgba[:, :w // 2]
    right = rgba[:, w // 2:]

    return crop_transparent(left), crop_transparent(right)


# 读取主图
img = cv2.imread(IMG_PATH)

if img is None:
    raise FileNotFoundError(f"找不到图片：{IMG_PATH}")


# 图片过大时自动缩小，避免窗口超过屏幕
h0, w0 = img.shape[:2]
scale = min(620 / w0, 560 / h0, 1.0)

img = cv2.resize(img, (int(w0 * scale), int(h0 * scale)))

H, W = img.shape[:2]


# 读取全部PNG贴图，并裁掉透明空白
GLASSES_RGBA = crop_transparent(load_rgba(GLASSES_PATH))
EARS_RGBA = crop_transparent(load_rgba(EARS_PATH))
MASK_RGBA = crop_transparent(load_rgba(MASK_PATH))
SCAR_RGBA = crop_transparent(load_rgba(SCAR_PATH))
STICKER_RGBA = crop_transparent(load_rgba(STICKER_PATH))

# 左右猫耳拆开
EARS_LEFT_RGBA, EARS_RIGHT_RGBA = split_ears(EARS_RGBA)


# ============================================================
# 3. 基础图片特效
# ============================================================

def gray(x):
    """灰度化：彩色图 -> 灰度图 -> 再转回3通道"""
    g = cv2.cvtColor(x, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)


def sharpen(x):
    """
    锐化。

    中间像素权重大，周围像素为负，
    会强化当前像素和周围像素之间的差异，
    因此边缘更加清晰。
    """
    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ], dtype=np.float32)

    return cv2.filter2D(x, -1, kernel)


def mosaic(x, block=18):
    """
    马赛克。

    先把图片缩小，主动丢失细节；
    再用最近邻插值放大，就会出现大色块。
    """
    h, w = x.shape[:2]

    small = cv2.resize(
        x,
        (max(1, w // block), max(1, h // block))
    )

    return cv2.resize(
        small,
        (w, h),
        interpolation=cv2.INTER_NEAREST
    )


def blur_effect(x):
    """高斯模糊，模拟雾化和柔焦"""
    return cv2.GaussianBlur(x, (21, 21), 0)


def emboss(x):
    """通过方向卷积制造浮雕效果"""
    kernel = np.array([
        [-2, -1, 0],
        [-1, 1, 1],
        [0, 1, 2]
    ], dtype=np.float32)

    y = cv2.filter2D(x, cv2.CV_32F, kernel)

    # 卷积后可能出现负数，因此整体加128恢复到正常亮度范围
    return np.clip(y + 128, 0, 255).astype(np.uint8)


def invert(x):
    """底片效果：每个像素做 255 - 原像素"""
    return 255 - x


def oil(x):
    """
    简化油画效果。

    双边滤波负责平滑颜色但尽量保留边缘，
    再减少颜色等级，使颜色呈现块状颜料感。
    """
    y = cv2.bilateralFilter(x, 9, 60, 60)
    return ((y // 32) * 32).astype(np.uint8)


# ============================================================
# 4. 高级调色和科技特效
# ============================================================

def cyber(x):
    """赛博朋克：加强蓝色和红色，降低绿色，并提高对比度"""
    b, g, r = cv2.split(x)

    b = cv2.add(b, 40)
    g = cv2.subtract(g, 20)
    r = cv2.add(r, 35)

    y = cv2.merge([b, g, r])

    return cv2.convertScaleAbs(y, alpha=1.25)


def vintage(x):
    """复古胶片：整体偏黄，并加入细小随机颗粒"""
    y = x.astype(np.float32)

    y[:, :, 0] *= 0.78
    y[:, :, 1] *= 0.96
    y[:, :, 2] *= 1.10

    noise = np.random.default_rng(1).normal(0, 7, y.shape)

    return np.clip(y + noise, 0, 255).astype(np.uint8)


def japanese(x):
    """日系清新：提高亮度、降低对比度，并略微偏青绿"""
    y = cv2.convertScaleAbs(x, alpha=0.85, beta=35)

    b, g, r = cv2.split(y)

    b = cv2.add(b, 10)
    g = cv2.add(g, 8)

    return cv2.merge([b, g, r])


def rgb_shift(x, distance=10):
    """RGB错位：红色向右、蓝色向左移动"""
    b, g, r = cv2.split(x)

    b = np.roll(b, -distance, axis=1)
    r = np.roll(r, distance, axis=1)

    return cv2.merge([b, g, r])


def crt(x):
    """周期性压暗横向像素行，模拟CRT扫描线"""
    y = x.copy()

    y[::4] = (y[::4] * 0.45).astype(np.uint8)
    y[1::4] = (y[1::4] * 0.70).astype(np.uint8)

    return y


def thermal(x):
    """
    模拟热成像视觉效果。

    注意这里只是伪彩色，
    并不是真的检测温度。
    """
    g = cv2.cvtColor(x, cv2.COLOR_BGR2GRAY)

    return cv2.applyColorMap(g, cv2.COLORMAP_JET)


# ============================================================
# 5. MediaPipe 人脸检测
# ============================================================

if not hasattr(mp, "solutions"):
    raise RuntimeError(
        "当前MediaPipe不支持mp.solutions，请使用 mediapipe==0.10.21"
    )


face_mesh_api = mp.solutions.face_mesh

face_mesh = face_mesh_api.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)


# MediaPipe使用RGB，因此先把OpenCV的BGR转成RGB
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

result = face_mesh.process(rgb)


# pts保存所有人脸关键点的像素坐标
pts = None

if result.multi_face_landmarks:

    face = result.multi_face_landmarks[0]

    pts = [
        (int(p.x * W), int(p.y * H))
        for p in face.landmark
    ]

else:
    print("没有检测到人脸，人脸相关效果会保持原图。")


def has_face():
    """判断是否成功检测到人脸"""
    return pts is not None


def P(index):
    """根据MediaPipe关键点编号取得像素坐标"""
    return pts[index]


def eye_centers():
    """
    获取左右眼中心。

    每只眼睛取两个眼角，
    两个眼角坐标求平均，
    得到一个比较稳定的眼睛中心。
    """
    left = (
        (P(33)[0] + P(133)[0]) // 2,
        (P(33)[1] + P(133)[1]) // 2
    )

    right = (
        (P(362)[0] + P(263)[0]) // 2,
        (P(362)[1] + P(263)[1]) // 2
    )

    return left, right


def eye_distance():
    """计算两眼中心之间的距离"""
    left, right = eye_centers()

    return math.hypot(
        right[0] - left[0],
        right[1] - left[1]
    )


def eye_angle():
    """
    根据两眼连线计算头部倾斜角。

    图像坐标系中Y轴向下，
    而OpenCV的旋转方向和数学坐标系不同，
    所以这里使用负号修正方向。
    """
    left, right = eye_centers()

    dx = right[0] - left[0]
    dy = right[1] - left[1]

    return -math.degrees(math.atan2(dy, dx))


def face_width():
    """根据左右脸边缘关键点估算脸宽"""
    left = np.float32(P(234))
    right = np.float32(P(454))

    return np.linalg.norm(right - left)


# ============================================================
# 6. 人脸蒙版
# ============================================================

def get_face_mask():
    """
    创建人脸区域蒙版。

    白色区域：脸
    黑色区域：背景

    可以用来实现：
    背景黑白、人脸彩色；
    只给人脸打马赛克。
    """
    mask = np.zeros((H, W), np.uint8)

    if not has_face():
        return mask

    ids = {
        i
        for edge in face_mesh_api.FACEMESH_FACE_OVAL
        for i in edge
    }

    face_points = np.array(
        [pts[i] for i in ids],
        dtype=np.int32
    )

    hull = cv2.convexHull(face_points)

    cv2.fillConvexPoly(mask, hull, 255)

    # 模糊边缘，让融合不那么生硬
    return cv2.GaussianBlur(mask, (15, 15), 0)


FACE_MASK = get_face_mask()


def blend(background, foreground, mask):
    """使用人脸蒙版混合两张图片"""
    alpha = (mask / 255.0)[:, :, None]

    y = (
        background * (1 - alpha)
        +
        foreground * alpha
    )

    return np.clip(y, 0, 255).astype(np.uint8)


# ============================================================
# 7. PNG贴图通用工具
# ============================================================

def resize_keep_ratio(rgba, target_w=None, target_h=None):
    """
    等比例缩放贴图。

    只指定目标宽度或目标高度其中之一，
    另一边按原始比例自动计算，
    因此不会把贴图压扁。
    """
    h, w = rgba.shape[:2]

    if target_w is not None:
        scale = target_w / w

    elif target_h is not None:
        scale = target_h / h

    else:
        return rgba.copy()

    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))

    return cv2.resize(
        rgba,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )


def rotate_rgba(rgba, angle):
    """
    旋转透明PNG，并自动扩大画布。

    如果画布不扩大，
    旋转后的猫耳、墨镜边缘可能被裁掉。
    """
    h, w = rgba.shape[:2]

    center = (w / 2, h / 2)

    M = cv2.getRotationMatrix2D(
        center,
        angle,
        1
    )

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)

    # 把旋转后的图案重新移到新画布中央
    M[0, 2] += new_w / 2 - center[0]
    M[1, 2] += new_h / 2 - center[1]

    return cv2.warpAffine(
        rgba,
        M,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )


def overlay_rgba(base, rgba, center, alpha_scale=1.0, mode="normal"):
    """
    把透明PNG叠加到原图。

    base：
        原图，BGR三通道

    rgba：
        透明PNG，BGRA四通道

    center：
        贴图中心坐标

    mode="normal"：
        普通透明叠加

    mode="multiply"：
        正片叠底，更适合战损、妆容等效果
    """
    out = base.copy()

    rh, rw = rgba.shape[:2]
    cx, cy = center

    x1 = int(cx - rw / 2)
    y1 = int(cy - rh / 2)

    x2 = x1 + rw
    y2 = y1 + rh

    # 计算贴图与原图真正重叠的区域
    bx1 = max(0, x1)
    by1 = max(0, y1)

    bx2 = min(base.shape[1], x2)
    by2 = min(base.shape[0], y2)

    # 完全在图片外面就直接返回
    if bx1 >= bx2 or by1 >= by2:
        return out

    # 得到对应PNG内部应该裁出来的区域
    sx1 = bx1 - x1
    sy1 = by1 - y1

    sx2 = sx1 + (bx2 - bx1)
    sy2 = sy1 + (by2 - by1)

    patch = rgba[sy1:sy2, sx1:sx2]

    # 前三个通道是BGR颜色
    fg = patch[:, :, :3].astype(np.float32)

    # 第四个通道是透明度
    alpha = patch[:, :, 3].astype(np.float32) / 255.0

    alpha *= alpha_scale

    # 从二维变成三维，方便和BGR三个颜色通道一起运算
    alpha = alpha[:, :, None]

    roi = out[by1:by2, bx1:bx2].astype(np.float32)

    if mode == "multiply":

        # 正片叠底公式
        multiplied = roi * fg / 255.0

        roi = (
            roi * (1 - alpha)
            +
            multiplied * alpha
        )

    else:

        # 普通alpha透明融合
        roi = (
            roi * (1 - alpha)
            +
            fg * alpha
        )

    out[by1:by2, bx1:bx2] = np.clip(
        roi,
        0,
        255
    ).astype(np.uint8)

    return out


# ============================================================
# 8. 人脸相关特效
# ============================================================

def face_color(x):
    """背景黑白，人脸保留彩色"""

    if not has_face():
        return x.copy()

    return blend(
        gray(x),
        x,
        FACE_MASK
    )


def face_mosaic(x):
    """只给人脸区域打马赛克"""

    if not has_face():
        return x.copy()

    return blend(
        x,
        mosaic(x, 12),
        FACE_MASK
    )


def face_points(x):
    """
    显示人脸框和关键点。

    这个功能主要用来展示：
    “程序确实完成了人脸检测和关键点定位”。
    """
    y = x.copy()

    if not has_face():
        return y

    ids = {
        i
        for edge in face_mesh_api.FACEMESH_FACE_OVAL
        for i in edge
    }

    face_pts = np.array(
        [pts[i] for i in ids]
    )

    x1, y1 = face_pts.min(axis=0)
    x2, y2 = face_pts.max(axis=0)

    cv2.rectangle(
        y,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    # 关键点很多，只画一部分，否则太密
    for p in pts[::4]:
        cv2.circle(
            y,
            p,
            1,
            (0, 255, 255),
            -1
        )

    return y


# ============================================================
# 9. 自动霓虹墨镜
# ============================================================

def add_glasses_asset(x):
    """
    墨镜定位方法：

    两眼中点 -> 墨镜中心位置
    两眼距离 -> 墨镜大小
    两眼连线 -> 墨镜旋转角度
    """

    if not has_face():
        return x.copy()

    left_eye, right_eye = eye_centers()

    distance = eye_distance()
    angle = eye_angle()

    # 两眼中点
    center_x = (left_eye[0] + right_eye[0]) // 2
    center_y = (left_eye[1] + right_eye[1]) // 2

    # 稍微向下调整，让镜片真正覆盖眼睛
    center_y += int(
        distance * GLASSES_Y_OFFSET
    )

    # 根据眼距控制墨镜大小
    target_w = int(
        distance * GLASSES_WIDTH_SCALE
    )

    rgba = resize_keep_ratio(
        GLASSES_RGBA,
        target_w=target_w
    )

    rgba = rotate_rgba(
        rgba,
        angle
    )

    return overlay_rgba(
        x,
        rgba,
        (center_x, center_y)
    )


# ============================================================
# 10. 左右猫耳独立定位
# ============================================================

def add_ears_asset(x):
    """
    猫耳定位是本版本重点。

    原来的问题：
    两个耳朵放在一整张PNG里，
    只能控制一个整体中心点。

    现在：
    左耳和右耳分别定位。

    算法：
    1. 两眼连线确定人物头部“横向方向”
    2. 垂直于两眼连线的方向就是“头顶方向”
    3. 以额头关键点为基准
    4. 分别向左、向右找到两只耳朵的位置
    5. 再沿头顶方向向上移动
    """

    if not has_face():
        return x.copy()

    fw = face_width()

    left_eye, right_eye = eye_centers()

    # 额头中央附近的关键点
    forehead = np.array(
        P(10),
        dtype=np.float32
    )

    # --------------------------------------------------------
    # 1. 计算两眼连线方向
    # --------------------------------------------------------

    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]

    length = math.hypot(dx, dy)

    if length < 1:
        length = 1

    # 横向单位向量
    ux = dx / length
    uy = dy / length

    # --------------------------------------------------------
    # 2. 计算“头顶方向”
    #
    # 如果人物正着：
    # 横向约为 (1,0)
    # 头顶方向就约为 (0,-1)
    # --------------------------------------------------------

    up_x = uy
    up_y = -ux

    # 两个耳朵左右分开的距离
    side = fw * EAR_SIDE_OFFSET

    # 耳朵整体向头顶方向移动
    up = fw * EAR_UP_OFFSET

    # --------------------------------------------------------
    # 3. 分别计算左右耳的底部锚点
    # --------------------------------------------------------

    left_base = np.array([
        forehead[0] - ux * side + up_x * up,
        forehead[1] - uy * side + up_y * up
    ])

    right_base = np.array([
        forehead[0] + ux * side + up_x * up,
        forehead[1] + uy * side + up_y * up
    ])

    # --------------------------------------------------------
    # 4. 每只耳朵单独缩放和旋转
    # --------------------------------------------------------

    ear_width = int(
        fw * EAR_WIDTH_SCALE
    )

    left_ear = resize_keep_ratio(
        EARS_LEFT_RGBA,
        target_w=ear_width
    )

    right_ear = resize_keep_ratio(
        EARS_RIGHT_RGBA,
        target_w=ear_width
    )

    angle = eye_angle()

    left_ear = rotate_rgba(
        left_ear,
        angle
    )

    right_ear = rotate_rgba(
        right_ear,
        angle
    )

    # --------------------------------------------------------
    # 5. 锚点代表“耳朵底部”，而overlay使用的是图片中心。
    #
    # 所以要把中心沿头顶方向移动约半个耳朵高度。
    #
    # EAR_BASE_DEPTH 用来让耳朵底部稍微埋进头发。
    # --------------------------------------------------------

    left_h = left_ear.shape[0]
    right_h = right_ear.shape[0]

    left_move = left_h * (0.5 - EAR_BASE_DEPTH)
    right_move = right_h * (0.5 - EAR_BASE_DEPTH)

    left_center = (
        int(left_base[0] + up_x * left_move),
        int(left_base[1] + up_y * left_move)
    )

    right_center = (
        int(right_base[0] + up_x * right_move),
        int(right_base[1] + up_y * right_move)
    )

    # --------------------------------------------------------
    # 6. 两只耳朵分别叠加
    # --------------------------------------------------------

    y = overlay_rgba(
        x,
        left_ear,
        left_center
    )

    y = overlay_rgba(
        y,
        right_ear,
        right_center
    )

    return y


# ============================================================
# 11. 赛博机械面罩
# ============================================================

def add_mask_asset(x):
    """
    面罩定位：

    鼻尖 -> 水平中心
    鼻尖和下巴的中间位置 -> 垂直中心
    脸宽 -> 面罩大小
    两眼连线 -> 面罩旋转
    """

    if not has_face():
        return x.copy()

    fw = face_width()

    nose = P(1)
    chin = P(152)

    angle = eye_angle()

    center_x = nose[0]

    center_y = int(
        (nose[1] + chin[1]) / 2
    )

    center_y += int(
        fw * MASK_Y_OFFSET
    )

    target_w = int(
        fw * MASK_WIDTH_SCALE
    )

    rgba = resize_keep_ratio(
        MASK_RGBA,
        target_w=target_w
    )

    rgba = rotate_rgba(
        rgba,
        angle
    )

    # 透明度不断周期变化，形成呼吸灯效果
    breathe = (
        math.sin(time.time() * 3)
        +
        1
    ) / 2

    alpha_scale = (
        0.82
        +
        0.18 * breathe
    )

    return overlay_rgba(
        x,
        rgba,
        (center_x, center_y),
        alpha_scale=alpha_scale
    )


# ============================================================
# 12. 腮红和战损
# ============================================================

def add_scar_asset(x):
    """在脸颊附近添加战损划痕"""

    if not has_face():
        return x.copy()

    fw = face_width()

    cheek = P(346)

    target_w = int(
        fw * SCAR_WIDTH_SCALE
    )

    rgba = resize_keep_ratio(
        SCAR_RGBA,
        target_w=target_w
    )

    rgba = rotate_rgba(
        rgba,
        eye_angle() - 10
    )

    center = (
        int(cheek[0] + fw * 0.03),
        int(cheek[1] - fw * 0.02)
    )

    # 使用正片叠底，让伤痕更像印在皮肤上
    return overlay_rgba(
        x,
        rgba,
        center,
        alpha_scale=0.85,
        mode="multiply"
    )


def makeup_plus_scar(x):
    """半透明腮红 + 战损"""

    y = x.copy()

    if not has_face():
        return y

    fw = abs(
        P(454)[0]
        -
        P(234)[0]
    )

    rx = max(
        8,
        int(fw * 0.10)
    )

    ry = max(
        5,
        int(fw * 0.05)
    )

    overlay = y.copy()

    # 左右脸颊
    for p in (
        P(117),
        P(346)
    ):

        cv2.ellipse(
            overlay,
            p,
            (rx, ry),
            0,
            0,
            360,
            (210, 180, 255),
            -1,
            cv2.LINE_AA
        )

    # 腮红只使用较低透明度
    y = cv2.addWeighted(
        overlay,
        0.22,
        y,
        0.78,
        0
    )

    return add_scar_asset(y)


# ============================================================
# 13. 自由贴纸
# ============================================================

sticker_pos = [
    W // 2,
    H // 2
]

sticker_scale = 1.0
sticker_angle = 0.0


def free_sticker(x):
    """
    自由贴纸支持：

    鼠标拖动位置
    鼠标滚轮缩放
    A / D旋转
    靠近鼻子时自动吸附
    """

    global sticker_pos

    base_h = int(
        H * STICKER_BASE_H_RATIO
    )

    target_h = max(
        40,
        int(base_h * sticker_scale)
    )

    rgba = resize_keep_ratio(
        STICKER_RGBA,
        target_h=target_h
    )

    rgba = rotate_rgba(
        rgba,
        sticker_angle
    )

    out = overlay_rgba(
        x,
        rgba,
        sticker_pos
    )

    # 自动吸附
    if has_face():

        anchor = P(1)

        distance = math.hypot(
            sticker_pos[0] - anchor[0],
            sticker_pos[1] - anchor[1]
        )

        if distance < 55:

            sticker_pos = [
                anchor[0],
                anchor[1]
            ]

            # 吸附时显示光晕
            glow = np.zeros_like(out)

            cv2.circle(
                glow,
                anchor,
                45,
                (255, 255, 0),
                10,
                cv2.LINE_AA
            )

            glow = cv2.GaussianBlur(
                glow,
                (0, 0),
                10
            )

            out = cv2.addWeighted(
                out,
                1,
                glow,
                0.8,
                0
            )

    return out


# ============================================================
# 14. 特效统一管理
# ============================================================

effects = {
    "Original": lambda x: x.copy(),

    "Gray": gray,
    "Sharpen": sharpen,
    "Mosaic": mosaic,
    "Blur": blur_effect,
    "Emboss": emboss,
    "Invert": invert,
    "Oil": oil,

    "Cyber": cyber,
    "Vintage": vintage,
    "Japanese": japanese,
    "RGB Shift": rgb_shift,
    "CRT": crt,
    "Thermal": thermal,

    "Face Color": face_color,
    "Face Mosaic": face_mosaic,
    "Face Points": face_points,

    "Glasses": add_glasses_asset,
    "Cat Ears": add_ears_asset,
    "Cyber Mask": add_mask_asset,
    "Makeup+Scar": makeup_plus_scar,
    "Sticker": free_sticker
}


effect_names = list(
    effects.keys()
)


# 这两个特效需要实时重新计算：
# Cyber Mask 要做呼吸动画
# Sticker 会跟随鼠标
dynamic_effects = {
    "Cyber Mask",
    "Sticker"
}


# ============================================================
# 15. UI尺寸参数
# ============================================================

COLS = 5

BUTTON_H = 34
GAP = 8
INFO_H = 30

ROWS = math.ceil(
    len(effect_names)
    /
    COLS
)

PANEL_H = (
    INFO_H
    +
    GAP
    +
    ROWS * (BUTTON_H + GAP)
    +
    GAP
)

UI_W = max(
    W,
    760
)

UI_H = (
    H
    +
    PANEL_H
)

# 图片水平居中
IMAGE_X = (
    UI_W - W
) // 2


# ============================================================
# 16. 当前状态变量
# ============================================================

current_effect = "Original"

effect_image = img.copy()

compare_mode = False

slider_x = (
    W // 2
)

scan_active = False
scan_y = 0

drag_slider = False
drag_sticker = False

button_rects = {}


# ============================================================
# 17. 特效切换
# ============================================================

def choose_effect(name):
    """切换当前特效"""

    global current_effect
    global effect_image
    global scan_active
    global scan_y

    current_effect = name

    # 静态特效只算一次，减少重复计算
    if name not in dynamic_effects:
        effect_image = effects[name](img)

    # 切换特效时播放扫描动画
    scan_active = (
        name != "Original"
    )

    scan_y = 0


def full_effect():
    """获取当前完整特效图"""

    if current_effect in dynamic_effects:
        return effects[current_effect](img)

    return effect_image.copy()


# ============================================================
# 18. 扫描过渡和左右对比
# ============================================================

def build_frame():
    """
    生成最终显示画面。

    优先级：
    扫描动画 > 左右对比 > 普通特效
    """

    effect = full_effect()

    # ---------- 扫描动画 ----------

    if scan_active:

        y = min(
            scan_y,
            H
        )

        frame = img.copy()

        # 扫描线以上替换成特效
        frame[:y] = effect[:y]

        if y < H:

            glow = frame.copy()

            cv2.line(
                glow,
                (0, y),
                (W, y),
                (255, 255, 0),
                8
            )

            frame = cv2.addWeighted(
                glow,
                0.45,
                frame,
                0.55,
                0
            )

            # 中间再画一条亮白线
            cv2.line(
                frame,
                (0, y),
                (W, y),
                (255, 255, 255),
                2
            )

        return frame


    # ---------- 左右拖拽对比 ----------

    if compare_mode:

        x = max(
            0,
            min(W, slider_x)
        )

        frame = effect.copy()

        # 左边恢复成原图
        frame[:, :x] = img[:, :x]

        # 分割线
        cv2.line(
            frame,
            (x, 0),
            (x, H),
            (255, 255, 255),
            2
        )

        # 分割线中间的拖动圆点
        cv2.circle(
            frame,
            (x, H // 2),
            11,
            (255, 255, 255),
            -1
        )

        cv2.circle(
            frame,
            (x, H // 2),
            7,
            (70, 70, 70),
            -1
        )

        return frame


    return effect


# ============================================================
# 19. 毛玻璃胶囊按钮
# ============================================================

def pill(canvas, x, y, w, h, selected=False):
    """
    胶囊按钮由：
    一个矩形 + 左右两个圆
    组成。

    最后使用 addWeighted 进行半透明融合。
    """

    radius = h // 2

    layer = canvas.copy()

    if selected:
        color = (255, 255, 255)
        alpha = 0.30
    else:
        color = (210, 210, 210)
        alpha = 0.13

    cv2.rectangle(
        layer,
        (x + radius, y),
        (x + w - radius, y + h),
        color,
        -1
    )

    cv2.circle(
        layer,
        (x + radius, y + radius),
        radius,
        color,
        -1
    )

    cv2.circle(
        layer,
        (x + w - radius, y + radius),
        radius,
        color,
        -1
    )

    canvas[:] = cv2.addWeighted(
        layer,
        alpha,
        canvas,
        1 - alpha,
        0
    )


# ============================================================
# 20. 绘制整个界面
# ============================================================

def draw_ui(frame):
    """绘制图片区域、毛玻璃背景和按钮"""

    global button_rects

    canvas = np.zeros(
        (UI_H, UI_W, 3),
        np.uint8
    )

    # 图片放在顶部中间
    canvas[
        :H,
        IMAGE_X:IMAGE_X + W
    ] = frame


    # ---------- 毛玻璃控制面板 ----------

    panel_src = frame[
        max(0, H // 2):
    ]

    panel = cv2.resize(
        panel_src,
        (UI_W, PANEL_H)
    )

    panel = cv2.GaussianBlur(
        panel,
        (31, 31),
        0
    )

    panel = cv2.addWeighted(
        panel,
        0.55,
        np.zeros_like(panel),
        0.45,
        0
    )

    canvas[H:] = panel


    # ---------- 操作提示 ----------

    text = (
        f"Effect: {current_effect}   "
        f"C: Compare {'ON' if compare_mode else 'OFF'}   "
        "S: Save   SPACE: Replay"
    )

    cv2.putText(
        canvas,
        text,
        (10, H + 21),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.48,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # ---------- 特效按钮 ----------

    button_w = (
        UI_W
        -
        2 * GAP
        -
        (COLS - 1) * GAP
    ) // COLS

    button_rects = {}

    for i, name in enumerate(
        effect_names
    ):

        row, col = divmod(
            i,
            COLS
        )

        x = GAP + col * (
            button_w + GAP
        )

        y = (
            H
            +
            INFO_H
            +
            GAP
            +
            row * (
                BUTTON_H + GAP
            )
        )

        selected = (
            name == current_effect
        )

        # 选中时按钮稍微上浮
        if selected:
            y -= 2

        pill(
            canvas,
            x,
            y,
            button_w,
            BUTTON_H,
            selected
        )

        cv2.putText(
            canvas,
            name,
            (x + 8, y + 22),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

        button_rects[name] = (
            x,
            y,
            button_w,
            BUTTON_H
        )


    # Sticker模式额外显示控制方式
    if current_effect == "Sticker":

        tip = (
            "Sticker: drag | wheel scale | "
            "A/D rotate | R reset"
        )

        cv2.putText(
            canvas,
            tip,
            (10, H - 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.43,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

    return canvas


# ============================================================
# 21. 鼠标交互
# ============================================================

def inside(x, y, rect):
    """判断鼠标是否位于某个按钮区域"""

    rx, ry, rw, rh = rect

    return (
        rx <= x <= rx + rw
        and
        ry <= y <= ry + rh
    )


def mouse(event, x, y, flags, param):
    """
    鼠标操作：

    点击按钮 -> 切换特效
    拖动对比线 -> 原图/特效比较
    拖动贴纸 -> 改变位置
    滚轮 -> 缩放贴纸
    """

    global drag_slider
    global drag_sticker
    global slider_x
    global sticker_pos
    global sticker_scale


    # ---------- 点击底部按钮 ----------

    if (
        event == cv2.EVENT_LBUTTONDOWN
        and
        y >= H
    ):

        for name, rect in button_rects.items():

            if inside(x, y, rect):

                choose_effect(name)

                return


    # ---------- 窗口坐标转换为图片内部坐标 ----------

    image_x = x - IMAGE_X
    image_y = y

    in_image = (
        0 <= image_x < W
        and
        0 <= image_y < H
    )


    # ---------- 滚轮缩放贴纸 ----------

    if (
        event == cv2.EVENT_MOUSEWHEEL
        and
        in_image
        and
        current_effect == "Sticker"
        and
        not compare_mode
    ):

        if flags > 0:
            sticker_scale += 0.08
        else:
            sticker_scale -= 0.08

        sticker_scale = float(
            np.clip(
                sticker_scale,
                0.15,
                1.6
            )
        )

        return


    # ---------- 鼠标按下 ----------

    if (
        event == cv2.EVENT_LBUTTONDOWN
        and
        in_image
    ):

        if compare_mode:

            drag_slider = True
            slider_x = image_x

        elif current_effect == "Sticker":

            drag_sticker = True

            sticker_pos = [
                image_x,
                image_y
            ]


    # ---------- 鼠标移动 ----------

    elif (
        event == cv2.EVENT_MOUSEMOVE
        and
        in_image
    ):

        if drag_slider:

            slider_x = image_x

        elif drag_sticker:

            sticker_pos = [
                image_x,
                image_y
            ]


    # ---------- 鼠标松开 ----------

    elif event == cv2.EVENT_LBUTTONUP:

        drag_slider = False
        drag_sticker = False


# ============================================================
# 22. 主循环
# ============================================================

WINDOW = "Photo Effects Demo"

cv2.namedWindow(
    WINDOW,
    cv2.WINDOW_AUTOSIZE
)

cv2.setMouseCallback(
    WINDOW,
    mouse
)


while True:

    # 得到当前图片
    frame = build_frame()

    # 加上UI
    ui = draw_ui(frame)

    # 显示
    cv2.imshow(
        WINDOW,
        ui
    )

    key = cv2.waitKey(20) & 0xFF


    # ---------- 扫描线不断往下走 ----------

    if scan_active:

        scan_y += 18

        if scan_y >= H:
            scan_active = False


    # ---------- Q / ESC：退出 ----------

    if key == ord("q") or key == 27:

        break


    # ---------- C：开启/关闭左右对比 ----------

    elif key == ord("c"):

        compare_mode = not compare_mode


    # ---------- S：保存结果 ----------

    elif key == ord("s"):

        cv2.imwrite(
            "result.jpg",
            full_effect()
        )

        print("已保存：result.jpg")


    # ---------- 空格：重新播放扫描动画 ----------

    elif (
        key == 32
        and
        current_effect != "Original"
    ):

        scan_active = True
        scan_y = 0


    # ---------- A：贴纸左旋 ----------

    elif (
        key == ord("a")
        and
        current_effect == "Sticker"
    ):

        sticker_angle -= 5


    # ---------- D：贴纸右旋 ----------

    elif (
        key == ord("d")
        and
        current_effect == "Sticker"
    ):

        sticker_angle += 5


    # ---------- R：重置贴纸 ----------

    elif (
        key == ord("r")
        and
        current_effect == "Sticker"
    ):

        sticker_pos = [
            W // 2,
            H // 2
        ]

        sticker_scale = 1.0
        sticker_angle = 0.0


# ============================================================
# 23. 释放资源
# ============================================================

cv2.destroyAllWindows()
face_mesh.close()

KeyboardInterrupt: 

In [2]:
import mediapipe as mp
import sys

print("Python:", sys.executable)
print("MediaPipe:", getattr(mp, "__version__", "unknown"))
print("路径:", mp.__file__)
print("有 solutions 吗:", hasattr(mp, "solutions"))

Python: d:\rag\.venv\Scripts\python.exe
MediaPipe: 0.10.35
路径: d:\rag\.venv\Lib\site-packages\mediapipe\__init__.py
有 solutions 吗: False
